In [2]:
# CELL 1 - Imports, tickers, data download (fixed for Adj Close)

import numpy as np
import pandas as pd
import yfinance as yf
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# 10 assets, 6 OHLCV features each = 60 total features
tickers = [
    "AAPL", "MSFT", "GOOG", "AMZN", "META",
    "TSLA", "NVDA", "JPM", "V", "UNH"
]

# Download daily OHLCV
# auto_adjust=False is REQUIRED to keep "Adj Close"
df = yf.download(
    tickers,
    start="2015-01-01",
    end="2025-01-01",
    auto_adjust=False
)

# Keep only OHLCV (6 features)
df = df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]

# Flatten multi-index columns: (feature, ticker) → "AAPL_Open"
df.columns = [f"{t}_{f}" for f, t in df.columns]

# Drop missing rows
df = df.dropna()

# Convert to float32 numpy array
data = df.values.astype(np.float32)

print("Data shape:", data.shape)   # (N, 60)

[*********************100%***********************]  10 of 10 completed


Data shape: (2516, 60)


In [3]:
# CELL 2 - Train/Val/Test split

N = len(data)

train_end = int(N * 0.8)
val_end   = int(N * 0.9)

train_raw = data[:train_end]
val_raw   = data[train_end:val_end]
test_raw  = data[val_end:]

print("Train:", train_raw.shape)
print("Val:  ", val_raw.shape)
print("Test: ", test_raw.shape)

Train: (2012, 60)
Val:   (252, 60)
Test:  (252, 60)


In [4]:
# CELL 3 - Scaling

scaler = StandardScaler()
scaler.fit(train_raw)

train_scaled = scaler.transform(train_raw)
val_scaled   = scaler.transform(val_raw)
test_scaled  = scaler.transform(test_raw)

print("Scaled shapes:", train_scaled.shape, val_scaled.shape, test_scaled.shape)

Scaled shapes: (2012, 60) (252, 60) (252, 60)


In [5]:
# CELL 4 - Windowed dataset and dataloaders

class WindowedDataset(Dataset):
    def __init__(self, data, input_len=96, output_len=24):
        super().__init__()
        self.data = data
        self.input_len = input_len
        self.output_len = output_len
        self.total_len = len(data)

        self.max_start = self.total_len - (input_len + output_len)
        if self.max_start <= 0:
            raise ValueError("Not enough data for given input_len + output_len.")

    def __len__(self):
        return self.max_start

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.input_len]                     # (96, 60)
        y = self.data[idx + self.input_len : idx + self.input_len + self.output_len]  # (24, 60)
        return torch.from_numpy(x), torch.from_numpy(y)


input_len = 96
output_len = 24
num_features = train_scaled.shape[1]  # should be 60

train_ds = WindowedDataset(train_scaled, input_len=input_len, output_len=output_len)
val_ds   = WindowedDataset(val_scaled,   input_len=input_len, output_len=output_len)
test_ds  = WindowedDataset(test_scaled,  input_len=input_len, output_len=output_len)

batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, drop_last=False)

len(train_loader), len(val_loader), len(test_loader)

(59, 5, 5)

In [15]:
# CELL 5 - CAAN-Full model (final corrected version)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class TemporalEncoder(nn.Module):
    def __init__(self, d_in, d_model=256, n_heads=8, num_layers=2, d_ff=512, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        x = self.proj(x)
        x = self.encoder(x)
        return x


class CrossAssetAttentionBlock(nn.Module):
    def __init__(self, d_model=256, n_heads=8, num_layers=2, d_ff=512, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_ff,
                dropout=dropout,
                batch_first=True
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):
        B, T, N, D = x.shape
        x = x.reshape(B * T, N, D)
        for layer in self.layers:
            x = layer(x)
        x = x.reshape(B, T, N, D)
        return x


class CAANFull(nn.Module):
    def __init__(
        self,
        input_len=96,
        output_len=24,
        num_assets=10,
        features_per_asset=6,
        d_model=256,
        n_heads=8,
        temp_layers=2,
        cross_layers=2,
        d_ff=512,
        dropout=0.1
    ):
        super().__init__()
        self.input_len = input_len
        self.output_len = output_len
        self.num_assets = num_assets
        self.features_per_asset = features_per_asset
        self.d_model = d_model

        assert num_assets * features_per_asset == 60

        self.temp_encoder = TemporalEncoder(
            d_in=features_per_asset,
            d_model=d_model,
            n_heads=n_heads,
            num_layers=temp_layers,
            d_ff=d_ff,
            dropout=dropout
        )

        self.cross_attn = CrossAssetAttentionBlock(
            d_model=d_model,
            n_heads=n_heads,
            num_layers=cross_layers,
            d_ff=d_ff,
            dropout=dropout
        )

        self.fusion_ln = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(num_assets * d_model, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, output_len * num_assets * features_per_asset)
        )

    def forward(self, x):
        B, T, F = x.shape

        x = x.reshape(B, T, self.num_assets, self.features_per_asset)

        x = x.permute(0, 2, 1, 3)
        x = x.reshape(B * self.num_assets, T, self.features_per_asset)

        x_temp = self.temp_encoder(x)
        x_temp = x_temp.reshape(B, self.num_assets, T, self.d_model)
        x_temp = x_temp.permute(0, 2, 1, 3)

        x_cross = self.cross_attn(x_temp)

        x_fused = self.fusion_ln(x_temp + x_cross)

        x_pooled = x_fused.mean(dim=1)  # (B, N_assets, d_model)

        x_flat = x_pooled.reshape(B, self.num_assets * self.d_model)

        out = self.head(x_flat)
        out = out.reshape(B, self.output_len, self.num_assets * self.features_per_asset)

        return out

In [16]:
# CELL 6 - Instantiate CAAN-Full, optimizer, scheduler, param count

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CAANFull(
    input_len=input_len,
    output_len=output_len,
    num_assets=10,
    features_per_asset=6,
    d_model=256,
    n_heads=8,
    temp_layers=2,
    cross_layers=2,
    d_ff=512,
    dropout=0.1
).to(device)

criterion_mse = nn.MSELoss()
criterion_mae = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 50
warmup_epochs = 3
max_lr = 1e-3
min_lr = 1e-6

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, (num_epochs - warmup_epochs - 1))
    return min_lr / max_lr + 0.5 * (1 - min_lr / max_lr) * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

Total parameters: 6209184
Trainable parameters: 6209184


In [17]:
# CELL 7 - Training and validation loop for CAAN-Full

best_val_mse = float("inf")
best_state_dict = None

for epoch in range(1, num_epochs + 1):
    model.train()
    train_mse = 0.0
    n_train_batches = 0

    current_lr = scheduler.get_last_lr()[0]

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion_mse(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_mse += loss.item()
        n_train_batches += 1

    train_mse /= max(1, n_train_batches)

    model.eval()
    val_mse = 0.0
    val_mae = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            preds = model(xb)
            mse = criterion_mse(preds, yb).item()
            mae = criterion_mae(preds, yb).item()

            val_mse += mse
            val_mae += mae
            n_val_batches += 1

    val_mse /= max(1, n_val_batches)
    val_mae /= max(1, n_val_batches)

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_state_dict = {k: v.cpu() for k, v in model.state_dict().items()}

    scheduler.step()

    print(f"===== Epoch {epoch}/{num_epochs} =====")
    print(f"LR:       {current_lr:.6f}")
    print(f"Train MSE:{train_mse:.6f}")
    print(f"Val   MSE:{val_mse:.6f}")
    print(f"Val   MAE:{val_mae:.6f}")

===== Epoch 1/50 =====
LR:       0.000333
Train MSE:0.275384
Val   MSE:1.661286
Val   MAE:0.919173
===== Epoch 2/50 =====
LR:       0.000667
Train MSE:0.171129
Val   MSE:1.368116
Val   MAE:0.825093
===== Epoch 3/50 =====
LR:       0.001000
Train MSE:0.168411
Val   MSE:1.401913
Val   MAE:0.829378
===== Epoch 4/50 =====
LR:       0.001000
Train MSE:0.149907
Val   MSE:1.275565
Val   MAE:0.776508
===== Epoch 5/50 =====
LR:       0.000999
Train MSE:0.139817
Val   MSE:1.488354
Val   MAE:0.843651
===== Epoch 6/50 =====
LR:       0.000995
Train MSE:0.141914
Val   MSE:1.408876
Val   MAE:0.848952
===== Epoch 7/50 =====
LR:       0.000990
Train MSE:0.131009
Val   MSE:1.457729
Val   MAE:0.827364
===== Epoch 8/50 =====
LR:       0.000981
Train MSE:0.130393
Val   MSE:1.299065
Val   MAE:0.776430
===== Epoch 9/50 =====
LR:       0.000971
Train MSE:0.127689
Val   MSE:1.233952
Val   MAE:0.750053
===== Epoch 10/50 =====
LR:       0.000959
Train MSE:0.125183
Val   MSE:1.520656
Val   MAE:0.847235
===== Epo

In [18]:
# CELL 8 - Load best CAAN-Full and evaluate on test set

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    model.to(device)
else:
    print("Warning: best_state_dict is None; using last-epoch weights.")

model.eval()
test_mse = 0.0
test_mae = 0.0
n_test_batches = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        preds = model(xb)
        mse = criterion_mse(preds, yb).item()
        mae = criterion_mae(preds, yb).item()

        test_mse += mse
        test_mae += mae
        n_test_batches += 1

test_mse /= max(1, n_test_batches)
test_mae /= max(1, n_test_batches)

print("===== CAAN-Full Test Performance =====")
print(f"Test MSE: {test_mse:.6f}")
print(f"Test MAE: {test_mae:.6f}")

===== CAAN-Full Test Performance =====
Test MSE: 16.463074
Test MAE: 2.169226
